# 05 — Question 3 : changements d'exposition et changements de salaire (within-person)

**Question :** Among workers observed in several waves, are changes in occupational AI exposure associated with changes in wages, once individual fixed characteristics, year effects and changes in occupational skill level are controlled for? How does this within-person association compare with the cross-sectional one?

**Règle :** aucune ligne individuelle affichée, seulement des coefficients et des comptages.

**Idée.** Q2 compare des personnes **différentes**. Si les métiers exposés paient mieux, c'est peut-être parce que des personnes plus qualifiées y travaillent.
Ici, on compare **chaque personne à elle-même** : tout ce qui est fixe chez elle (talent, origine, qualité du diplôme) disparaît.
Le lien est alors identifié **uniquement par les personnes qui changent de métier**.

| Modèle | Comparaison | Ce qu'on enlève |
|---|---|---|
| 1. Pooled OLS | entre personnes (comme Q2) | rien |
| 2. Effets fixes individuels | chaque personne avec elle-même | tout ce qui est fixe chez la personne |
| 3. Effets fixes + grand groupe ISCO | idem, à niveau de qualification égal | l'effet des promotions / changements de niveau |

Les trois modèles sont estimés sur les **diplômés du tertiaire** (spécification principale, cohérente avec la question de recherche), puis sur **tous les salariés** (robustesse).

**Méthodes du cours :** `groupby().transform()` (Week 2), `get_dummies` (Week 2), `LinearRegression` (Week 6).
**Au-delà du cours (signalé) :** les effets fixes individuels et `statsmodels` pour les erreurs standard clusterisées par personne.

In [ ]:
# Si statsmodels manque : lancer une fois  %pip install statsmodels  puis Restart
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

DATA_PATH = "../Data/private/shp_isco4_ilo.parquet"

## 1. Préparer l'échantillon
- On garde seulement les **codes précis** (`exposure_source == "isco4"`) : les codes vagues (`2000`, `2500`…) créent de faux changements de métier.
- On garde les personnes observées **au moins 2 ans** : avec une seule observation, aucune comparaison avec soi-même n'est possible.
- `expo10` = exposition × 10 : le coefficient se lit alors comme l'effet d'une hausse de **0,1** de l'exposition, soit l'ordre de grandeur d'un changement de métier typique (médiane observée : 0,09).

In [ ]:
df = pd.read_parquet(DATA_PATH)

def preparer(d):
    d = d[(d["exposure_source"] == "isco4") & (d["sex"].isin([1, 2]))].copy()
    n_annees = d.groupby("idpers")["year"].transform("nunique")
    d = d[n_annees >= 2].copy()

    d["expo10"] = d["exposure_main"] * 10
    d["age2"] = d["age"] ** 2
    d["isco1"] = d["isco4"].str[:1]
    d["isco2"] = d["isco4"].str[:2]
    for col in ["sex", "isced", "region", "year"]:
        d[col] = d[col].astype("Int64").astype(str)
    return d

echantillons = {
    "tertiaires": preparer(df[df["tertiary"] == 1]),
    "tous": preparer(df),
}

for nom, d in echantillons.items():
    chg_isco2 = (d.groupby("idpers")["isco2"].nunique() > 1).sum()
    chg_expo = (d.groupby("idpers")["expo10"].nunique() > 1).sum()
    print(f"{nom:>10} : {len(d)} lignes | {d['idpers'].nunique()} personnes | "
          f"{chg_isco2} changent d'isco2 | {chg_expo} ont une exposition qui change")

## 2. Effets fixes : soustraire la moyenne de chaque personne
L'effet fixe $\alpha_i$ capte tout ce qui est constant chez la personne $i$. Dans le modèle
$$\log w_{it} = \beta\,\text{expo}_{it} + \gamma_t + \alpha_i + X_{it}'\delta + \varepsilon_{it},$$
on retire à chaque variable sa **moyenne propre à la personne**. $\alpha_i$ disparaît, puisqu'une constante moins sa moyenne vaut 0.
Il ne reste que les **écarts de chaque personne par rapport à sa propre moyenne**. C'est l'estimateur *within*.

Conséquences :
- les variables fixes (sexe, niveau de formation) disparaissent aussi : on ne peut plus les estimer ;
- **l'âge (linéaire) est retiré** : pour une même personne, l'âge augmente exactement comme l'année, il est donc confondu avec les dummies d'année. On garde l'âge au carré.

In [ ]:
def demean(X, groupes):
    return X - X.groupby(groupes).transform("mean")

## 3. Estimer les trois modèles
Erreurs standard **clusterisées par personne** : les observations d'une même personne ne sont pas indépendantes.
`coef_%` = variation du salaire en % pour une hausse de 0,1 de l'exposition (approximation log).

In [ ]:
def estimer(d, nom_echantillon):
    y = d["log_income"]
    annees = pd.get_dummies(d["year"], prefix="y", drop_first=True, dtype=int)
    fixes = pd.get_dummies(d[["sex", "isced", "region"]], drop_first=True, dtype=int)
    groupe = pd.get_dummies(d["isco1"], prefix="isco1", drop_first=True, dtype=int)
    clusters = d["idpers"]

    specs = {
        "1_pooled_OLS": (pd.concat([d[["expo10", "age", "age2"]], annees, fixes], axis=1), False),
        "2_effets_fixes": (pd.concat([d[["expo10", "age2"]], annees], axis=1), True),
        "3_EF_+_grand_groupe": (pd.concat([d[["expo10", "age2"]], annees, groupe], axis=1), True),
    }

    lignes = []
    for nom, (X, effets_fixes) in specs.items():
        if effets_fixes:
            X_est, y_est = demean(X, clusters), demean(y, clusters)
        else:
            X_est, y_est = sm.add_constant(X), y
        res = sm.OLS(y_est, X_est).fit(cov_type="cluster", cov_kwds={"groups": clusters})
        b, se = res.params["expo10"], res.bse["expo10"]
        lignes.append({"echantillon": nom_echantillon, "modele": nom,
                       "coef_%": 100 * b, "IC95_bas_%": 100 * (b - 1.96 * se),
                       "IC95_haut_%": 100 * (b + 1.96 * se), "p_value": res.pvalues["expo10"],
                       "n_lignes": int(res.nobs), "n_personnes": clusters.nunique()})
    return lignes

resultats = []
for nom, d in echantillons.items():
    resultats += estimer(d, nom)
resultats = pd.DataFrame(resultats).set_index(["echantillon", "modele"])
resultats.round(3)

**Lien avec le cours :** le coefficient des effets fixes est exactement celui d'une `LinearRegression` (Week 6) sur les données « démoyennées ». `statsmodels` ne sert qu'à obtenir les erreurs standard clusterisées.

In [ ]:
d = echantillons["tertiaires"]
annees = pd.get_dummies(d["year"], prefix="y", drop_first=True, dtype=int)
X_dm = demean(pd.concat([d[["expo10", "age2"]], annees], axis=1), d["idpers"])
y_dm = demean(d["log_income"], d["idpers"])

lr = LinearRegression(fit_intercept=False).fit(X_dm, y_dm)
print("Coefficient expo10 (LinearRegression) :", round(100 * lr.coef_[0], 3), "%")
print("Coefficient expo10 (statsmodels)      :", round(resultats.loc[("tertiaires", "2_effets_fixes"), "coef_%"], 3), "%")

## 4. Lecture des résultats

| Comparaison | Si… | Alors… |
|---|---|---|
| **1 → 2** | le coefficient chute fortement | le lien vu en coupe (Q2) vient surtout de **qui** travaille dans les métiers exposés (sélection) |
| | le coefficient reste proche | le lien ne s'explique pas par des caractéristiques fixes des personnes |
| **2 → 3** | le coefficient disparaît | le lien passe par le **niveau de qualification** du métier (changements de grand groupe, promotions) |
| | le coefficient reste | à niveau de qualification égal, passer à un métier plus exposé reste associé au salaire |
| **tertiaires vs tous** | même sens | résultat robuste à l'échantillon |

Un intervalle de confiance qui contient 0 signifie qu'on ne peut pas distinguer l'association de zéro.

**Limites à mentionner :**
1. **Ce n'est pas causal.** Les effets fixes enlèvent ce qui est **fixe**, pas ce qui change **en même temps** que le métier (promotion, changement d'employeur, reconversion).
2. **Le bruit de codage** : une partie des changements de code restants peut encore être du bruit de codage, ce qui pousse le coefficient vers 0 (biais d'atténuation).
3. **Les changeurs ne sont pas représentatifs** : le coefficient within décrit les personnes qui changent de métier, pas l'ensemble des salariés.
4. **L'indice d'exposition est fixe** (OIT 2025) : l'exposition d'une personne ne change que si son métier change. On ne mesure pas l'arrivée de l'IA dans le temps.

## 5. Conclusion

| Sample | Model | Coef. (% per +0.1 exposure) | 95% CI | p |
|---|---|---|---|---|
| Tertiary | 1. Pooled OLS | +2.6 | [1.8; 3.3] | < 0.001 |
| | 2. Individual FE | +2.0 | [−0.1; 4.2] | 0.061 |
| | 3. FE + major group | +2.5 | [−0.1; 5.1] | 0.055 |
| All workers | 1. Pooled OLS | +3.7 | [3.2; 4.2] | < 0.001 |
| | 2. Individual FE | +1.2 | [−0.0; 2.5] | 0.058 |
| | 3. FE + major group | +2.0 | [0.1; 3.9] | 0.035 |

Identification comes from 692 tertiary graduates (1,311 workers overall) whose occupational exposure changes over the period.

**Answer to Q3.** Changes in occupational exposure are positively but weakly associated with wage changes within individuals.

- **Sorting.** Among all workers, the within-person association (+1.2% per 0.1 increase in exposure) is about one third of the cross-sectional one (+3.7%). Most of the cross-sectional link therefore reflects which workers sort into exposed occupations, rather than the occupation itself. Among tertiary graduates, sorting matters less (+2.6% pooled vs +2.0% within), as expected in a sample with more homogeneous education.
- **Skill level.** Controlling for changes in major occupational group (1-digit ISCO) raises the within estimate (+2.0% among all workers, 95% CI [0.1; 3.9]). A plausible explanation is that moves into clerical occupations (major group 4), which are the most exposed to generative AI but relatively low-paid, mask a positive association at constant skill level. This interpretation is a hypothesis and was not tested directly.
- **Link with Q2.** The two results are consistent. A wage difference of about 2% per 0.1 of exposure is statistically detectable within individuals, but too small to reduce prediction error noticeably, given a typical prediction error of about 36% (Q2).

**Limitations.** (1) Within-person estimates are imprecise: only one coefficient is significant at the 5% level, so results should be read as suggestive. (2) Part of the remaining occupation changes may be coding noise, which biases fixed-effects estimates towards zero. (3) Fixed effects remove time-invariant characteristics, but not events that change together with the occupation (promotions, change of employer), so the estimates are associations, not causal effects. (4) Occupation changers are not a random subset of workers. (5) The exposure index is fixed (ILO 2025): exposure changes only when the occupation changes, so these results describe wage differences across occupations, not the arrival of AI over time.

Intuitively, when a worker moves down from professional to clerical work, exposure rises while wages fall because of the change in skill level, not because of exposure itself; controlling for major group attributes this wage drop to the group change rather than to exposure.